In [ ]:
# ============================================
# PARTIE 1 : IMPORTATION DES BIBLIOTHÈQUES
# ============================================
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
import warnings
warnings.filterwarnings('ignore')

# Configuration
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

print(f"TensorFlow version: {tf.__version__}")

# ============================================
# PARTIE 2 : CHARGEMENT ET EXPLORATION DES DONNÉES
# ============================================
# Chargement du dataset MNIST
(X_train, y_train), (X_test, y_test) = mnist.load_data()

print("=" * 50)
print("INFORMATIONS SUR LE DATASET MNIST")
print("=" * 50)
print(f"Shape de X_train: {X_train.shape}")
print(f"Shape de X_test: {X_test.shape}")
print(f"Nombre de classes: {len(np.unique(y_train))}")
print(f"Valeurs des pixels: {X_train.min()} à {X_train.max()}")

# Visualisation d'exemples
fig, axes = plt.subplots(2, 5, figsize=(12, 6))
for i, ax in enumerate(axes.flat):
    ax.imshow(X_train[i], cmap='gray')
    ax.set_title(f'Chiffre: {y_train[i]}')
    ax.axis('off')
plt.suptitle('Exemples du dataset MNIST', fontsize=16)
plt.tight_layout()
plt.show()

# Distribution des classes
plt.figure(figsize=(10, 5))
unique, counts = np.unique(y_train, return_counts=True)
plt.bar(unique, counts, color='skyblue', edgecolor='black')
plt.title('Distribution des chiffres dans l\'ensemble d\'entraînement')
plt.xlabel('Chiffre')
plt.ylabel('Nombre d\'exemples')
plt.grid(True, alpha=0.3)
plt.show()

# ============================================
# PARTIE 3 : PRÉTRAITEMENT DES DONNÉES
# ============================================
# Normalisation des pixels (0-1)
X_train_norm = X_train.astype('float32') / 255.0
X_test_norm = X_test.astype('float32') / 255.0

# Encodage des labels en one-hot
y_train_cat = to_categorical(y_train, 10)
y_test_cat = to_categorical(y_test, 10)

print("Données prétraitées:")
print(f"X_train_norm shape: {X_train_norm.shape}")
print(f"y_train_cat shape: {y_train_cat.shape}")

# ============================================
# PARTIE 4 : MODÈLE FULLY CONNECTED (FCN)
# ============================================
def create_fcn_model(input_shape=(28, 28)):
    """
    Crée un réseau de neurones entièrement connecté pour MNIST
    """
    model = models.Sequential([
        # Aplatir l'image 28x28 en vecteur 784
        layers.Flatten(input_shape=input_shape),
        
        # Première couche cachée
        layers.Dense(512, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        
        # Deuxième couche cachée
        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        
        # Troisième couche cachée
        layers.Dense(128, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.2),
        
        # Couche de sortie (10 classes)
        layers.Dense(10, activation='softmax')
    ])
    
    return model

# Création et compilation du modèle FCN
fcn_model = create_fcn_model()
fcn_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("\n" + "=" * 50)
print("ARCHITECTURE DU MODÈLE FULLY CONNECTED")
print("=" * 50)
fcn_model.summary()

# ============================================
# PARTIE 5 : MODÈLE CNN
# ============================================
def create_cnn_model(input_shape=(28, 28, 1)):
    """
    Crée un réseau de neurones convolutif pour MNIST
    """
    model = models.Sequential([
        # Redimensionner pour ajouter le canal (28,28,1)
        layers.Reshape((28, 28, 1), input_shape=(28, 28)),
        
        # Bloc de convolution 1
        layers.Conv2D(32, (3, 3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.Conv2D(32, (3, 3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),
        
        # Bloc de convolution 2
        layers.Conv2D(64, (3, 3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.Conv2D(64, (3, 3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),
        
        # Bloc de convolution 3
        layers.Conv2D(128, (3, 3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.3),
        
        # Partie dense
        layers.Flatten(),
        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        
        layers.Dense(128, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        
        # Couche de sortie
        layers.Dense(10, activation='softmax')
    ])
    
    return model

# Création et compilation du modèle CNN
cnn_model = create_cnn_model()
cnn_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("\n" + "=" * 50)
print("ARCHITECTURE DU MODÈLE CNN")
print("=" * 50)
cnn_model.summary()

# ============================================
# PARTIE 6 : CALLBACKS POUR L'ENTRAÎNEMENT
# ============================================
def get_callbacks(model_name):
    """
    Définit les callbacks pour l'entraînement
    """
    return [
        callbacks.EarlyStopping(
            monitor='val_loss',
            patience=7,
            restore_best_weights=True,
            verbose=1
        ),
        callbacks.ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=3,
            min_lr=1e-6,
            verbose=1
        ),
        callbacks.ModelCheckpoint(
            f'best_{model_name}.h5',
            monitor='val_accuracy',
            save_best_only=True,
            verbose=1
        )
    ]

# ============================================
# PARTIE 7 : ENTRAÎNEMENT DES MODÈLES
# ============================================
EPOCHS = 30
BATCH_SIZE = 128

# Entraînement du modèle FCN
print("\n" + "=" * 50)
print("ENTRAÎNEMENT DU MODÈLE FULLY CONNECTED")
print("=" * 50)

fcn_history = fcn_model.fit(
    X_train_norm, y_train_cat,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_split=0.15,
    callbacks=get_callbacks('fcn'),
    verbose=1
)

# Entraînement du modèle CNN
print("\n" + "=" * 50)
print("ENTRAÎNEMENT DU MODÈLE CNN")
print("=" * 50)

cnn_history = cnn_model.fit(
    X_train_norm, y_train_cat,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_split=0.15,
    callbacks=get_callbacks('cnn'),
    verbose=1
)

# ============================================
# PARTIE 8 : VISUALISATION DES PERFORMANCES
# ============================================
def plot_training_history(history, model_name):
    """
    Trace les courbes de précision et de perte
    """
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    
    # Précision
    ax1.plot(history.history['accuracy'], label='Entraînement')
    ax1.plot(history.history['val_accuracy'], label='Validation')
    ax1.set_title(f'{model_name} - Précision')
    ax1.set_xlabel('Époque')
    ax1.set_ylabel('Précision')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Perte
    ax2.plot(history.history['loss'], label='Entraînement')
    ax2.plot(history.history['val_loss'], label='Validation')
    ax2.set_title(f'{model_name} - Perte')
    ax2.set_xlabel('Époque')
    ax2.set_ylabel('Perte')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

# Afficher les courbes
plot_training_history(fcn_history, 'Fully Connected')
plot_training_history(cnn_history, 'CNN')

# ============================================
# PARTIE 9 : ÉVALUATION DES MODÈLES
# ============================================
def evaluate_model(model, X_test, y_test_cat, y_test, model_name):
    """
    Évalue un modèle et affiche les métriques
    """
    # Prédictions
    y_pred_prob = model.predict(X_test)
    y_pred = np.argmax(y_pred_prob, axis=1)
    
    # Métriques
    loss, accuracy = model.evaluate(X_test, y_test_cat, verbose=0)
    
    print(f"\n{'='*50}")
    print(f"ÉVALUATION DU MODÈLE {model_name.upper()}")
    print(f"{'='*50}")
    print(f"Perte (Loss): {loss:.4f}")
    print(f"Précision (Accuracy): {accuracy:.4f}")
    
    # Matrice de confusion
    cm = confusion_matrix(y_test, y_pred)
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=range(10), yticklabels=range(10))
    plt.title(f'Matrice de Confusion - {model_name.upper()}')
    plt.xlabel('Prédictions')
    plt.ylabel('Vraies Valeurs')
    plt.tight_layout()
    plt.show()
    
    # Rapport de classification
    print("\nRapport de classification:")
    print(classification_report(y_test, y_pred))
    
    # Erreurs par classe
    class_errors = {}
    for i in range(10):
        mask = (y_test == i)
        errors = np.sum(y_pred[mask] != y_test[mask])
        total = np.sum(mask)
        class_errors[i] = errors / total if total > 0 else 0
    
    print("\nErreurs par chiffre:")
    for digit, error_rate in class_errors.items():
        print(f"Chiffre {digit}: {error_rate:.2%}")
    
    return y_pred

# Évaluation des deux modèles
print("\n" + "=" * 50)
print("ÉVALUATION DU MODÈLE FULLY CONNECTED")
print("=" * 50)
fcn_predictions = evaluate_model(fcn_model, X_test_norm, y_test_cat, y_test, 'FCN')

print("\n" + "=" * 50)
print("ÉVALUATION DU MODÈLE CNN")
print("=" * 50)
cnn_predictions = evaluate_model(cnn_model, X_test_norm, y_test_cat, y_test, 'CNN')

# ============================================
# PARTIE 10 : COMPARAISON DES MODÈLES
# ============================================
def compare_models(fcn_history, cnn_history):
    """
    Compare les performances des deux modèles
    """
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    # Comparaison de la précision
    axes[0].plot(fcn_history.history['val_accuracy'], label='FCN', color='blue')
    axes[0].plot(cnn_history.history['val_accuracy'], label='CNN', color='red')
    axes[0].set_title('Comparaison de la précision de validation')
    axes[0].set_xlabel('Époque')
    axes[0].set_ylabel('Précision')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Comparaison de la perte
    axes[1].plot(fcn_history.history['val_loss'], label='FCN', color='blue')
    axes[1].plot(cnn_history.history['val_loss'], label='CNN', color='red')
    axes[1].set_title('Comparaison de la perte de validation')
    axes[1].set_xlabel('Époque')
    axes[1].set_ylabel('Perte')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Métriques finales
    fcn_val_acc = max(fcn_history.history['val_accuracy'])
    cnn_val_acc = max(cnn_history.history['val_accuracy'])
    
    print("\n" + "=" * 50)
    print("COMPARAISON DES PERFORMANCES")
    print("=" * 50)
    print(f"FCN - Meilleure précision de validation: {fcn_val_acc:.4f}")
    print(f"CNN - Meilleure précision de validation: {cnn_val_acc:.4f}")
    print(f"Amélioration avec CNN: {(cnn_val_acc - fcn_val_acc)*100:.2f}%")

# Comparaison
compare_models(fcn_history, cnn_history)

# ============================================
# PARTIE 11 : VISUALISATION DES ERREURS
# ============================================
def visualize_errors(X_test, y_test, y_pred, model_name, num_examples=10):
    """
    Visualise les erreurs de classification
    """
    # Trouver les indices où la prédiction est fausse
    error_indices = np.where(y_test != y_pred)[0]
    print(f"Nombre total d'erreurs: {len(error_indices)}")
    
    if len(error_indices) == 0:
        print("Aucune erreur !")
        return
    
    # Sélectionner quelques exemples d'erreurs
    sample_indices = np.random.choice(error_indices, 
                                     min(num_examples, len(error_indices)), 
                                     replace=False)
    
    fig, axes = plt.subplots(2, 5, figsize=(15, 6))
    for i, ax in enumerate(axes.flat):
        if i < len(sample_indices):
            idx = sample_indices[i]
            ax.imshow(X_test[idx], cmap='gray')
            ax.set_title(f'Vrai: {y_test[idx]}\nPrédit: {y_pred[idx]}', 
                        color='red' if y_test[idx] != y_pred[idx] else 'green')
        ax.axis('off')
    
    plt.suptitle(f'Erreurs de classification - {model_name.upper()}', fontsize=16)
    plt.tight_layout()
    plt.show()

# Visualisation des erreurs pour le CNN
visualize_errors(X_test, y_test, cnn_predictions, 'CNN')

# ============================================
# PARTIE 12 : SAUVEGARDE DES MODÈLES
# ============================================
# Sauvegarde des modèles
fcn_model.save('mnist_fcn_model.h5')
cnn_model.save('mnist_cnn_model.h5')
print("\nModèles sauvegardés:")
print("- mnist_fcn_model.h5")
print("- mnist_cnn_model.h5")

# ============================================
# RÉSUMÉ FINAL
# ============================================
print("\n" + "=" * 50)
print("RÉSUMÉ DU PROJET")
print("=" * 50)
print(f"""
✅ Classification MNIST réalisée avec succès
✅ Deux modèles implémentés:
   - Fully Connected Network (FCN)
   - Convolutional Neural Network (CNN)
✅ Performance FCN: {max(fcn_history.history['val_accuracy'])*100:.2f}%
✅ Performance CNN: {max(cnn_history.history['val_accuracy'])*100:.2f}%
✅ Les deux modèles sont sauvegardés pour utilisation future
""")

: 